In [0]:
%run ../00-common/config

In [0]:
%run ./00_silver_helpers

In [0]:
from pyspark.sql import functions as F

sellers = spark.table(f"{catalog_name}.{bronze_schema}.sellers").drop("ingestion_timestamp", "source_file")

# a ka qytete me emra jo te vertete?
sellers.filter(F.col("seller_city").rlike("^[0-9]+$")) \
    .select("seller_id", "seller_zip_code_prefix", "seller_city", "seller_state").show(truncate=False)

In [0]:
# kontrolloj qytetin e vertete te geolocation
spark.table(f"{catalog_name}.{bronze_schema}.geolocation") \
    .filter(F.col("geolocation_zip_code_prefix") == "22790") \
    .select("geolocation_city", "geolocation_state").distinct().show(truncate=False)

In [0]:
# nje qytet per zip nga geolocation
geo_city = (spark.table(f"{catalog_name}.{bronze_schema}.geolocation")
    .groupBy("geolocation_zip_code_prefix")
    .agg(F.first("geolocation_city").alias("geo_city")))

sellers_silver = (
    sellers.join(
        geo_city,
        sellers.seller_zip_code_prefix == geo_city.geolocation_zip_code_prefix,
        how="left"
    )
    # qytet numerik ateher merr nga geolocation sipas zip
    .withColumn("seller_city",
        F.when(F.col("seller_city").rlike("^[0-9]+$"), F.col("geo_city"))
         .otherwise(F.col("seller_city")))
    .drop("geo_city", "geolocation_zip_code_prefix")
)

write_to_silver(sellers_silver, "sellers", catalog_name, silver_schema)

In [0]:
# check a u rregullua
spark.table(f"{catalog_name}.{silver_schema}.sellers") \
    .filter(F.col("seller_id") == "ceb7b4fb9401cd378de7886317ad1b47") \
    .select("seller_id", "seller_city", "seller_state").show(truncate=False)

In [0]:
from pyspark.sql import functions as F

# ==== 1. payment_type = not_defined? ====
print("=== payment_type ===")
spark.table(f"{catalog_name}.{bronze_schema}.order_payments") \
    .groupBy("payment_type").count().orderBy("count", ascending=False).show()

# ==== 2. geolocation: zip me shume qytete? ====
print("=== geolocation zip me >1 qytet ===")
g = spark.table(f"{catalog_name}.{bronze_schema}.geolocation")
mc = (g.groupBy("geolocation_zip_code_prefix")
    .agg(F.countDistinct("geolocation_city").alias("n"))
    .filter(F.col("n") > 1))
print("Zip me >1 qytet:", mc.count(), "nga", g.select("geolocation_zip_code_prefix").distinct().count())

# ==== 3. review_score jashte 1-5? ====
print("=== review_score jashte 1-5 ===")
spark.table(f"{catalog_name}.{bronze_schema}.order_reviews") \
    .filter(~F.col("review_score").isin("1","2","3","4","5") & F.col("review_score").isNotNull()) \
    .count()  # duhet 0 ose pak

# ==== 4. price/freight negativ ose zero? ====
print("=== order_items price/freight <= 0 ===")
oi = spark.table(f"{catalog_name}.{bronze_schema}.order_items")
print("price <= 0:", oi.filter(F.col("price") <= 0).count())
print("freight < 0:", oi.filter(F.col("freight_value") < 0).count())

# ==== 5. payment_value negativ ose zero? ====
print("=== payment_value <= 0 ===")
op = spark.table(f"{catalog_name}.{bronze_schema}.order_payments")
print("payment_value <= 0:", op.filter(F.col("payment_value") <= 0).count())

# ==== 6. order_status: cilat vlera? ====
print("=== order_status ===")
spark.table(f"{catalog_name}.{bronze_schema}.orders") \
    .groupBy("order_status").count().orderBy("count", ascending=False).show()

# ==== 7. shipping_limit_date para purchase? (logjike kohore) ====
print("=== order_items shipping para... (skip nese s'ka join, e leme per gold) ===")

In [0]:
from pyspark.sql import functions as F
g = spark.table(f"{catalog_name}.{bronze_schema}.geolocation")

# shembull: nje zip me shume qytete, si duken?
sample_zip = (g.groupBy("geolocation_zip_code_prefix")
    .agg(F.countDistinct("geolocation_city").alias("n"))
    .filter(F.col("n") > 1).orderBy(F.desc("n")).first()["geolocation_zip_code_prefix"])

print(f"Zip shembull: {sample_zip}")
g.filter(F.col("geolocation_zip_code_prefix") == sample_zip) \
    .groupBy("geolocation_city").count().orderBy(F.desc("count")).show(truncate=False)